# ClaimMitra - Data Protection & Privacy at Rest

**Module 5, Day 2 | 28 October 2026 | Dr. Deepali Salwan**
*AI Agent Security & Governance - Lloyds Bank*

---

### Five layers, five different attackers

| Layer | Stops | Built in |
|---|---|---|
| 1. Row-Level Security | The legitimate user who asks for rows that are not theirs | Parts B, C |
| 2. Field-Level Encryption | The DBA, the stolen snapshot, the stray log line | Part D |
| 3. Key Management | The leaked config file that would otherwise be a data breach | Part D |
| 4. Tamper-Evident Audit | The attacker who would rather you could not prove it | Part E, Challenge |
| 5. Compliance Mapping | The regulator, who asks for evidence rather than architecture | Part F |

### Lab plan

| Part | Topic | Time |
|---|---|---|
| A | Schema, roles and the access matrix | 10 min |
| B | RLS via filtered views + session context | 15 min |
| C | Write enforcement with INSTEAD OF triggers | 10 min |
| D | Field encryption, envelope keys, blind index | 15 min |
| E | Tamper-evident audit log - nine mandatory fields | 15 min |
| F | Agent scoping, PII scanning, UK GDPR & EU AI Act mapping | 10 min |
| * | Challenge - chain verification and tamper detection | 20 min |

> Each layer assumes the one above it has already failed. That is the whole
> point of defence in depth - no single control is load-bearing.

---

## Setup

Only one external package. Everything else is the standard library.

In [3]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                       'cryptography', '--quiet'])
print('cryptography installed.')

cryptography installed.


In [4]:
import sqlite3, hashlib, hmac, json, os, base64, datetime, re
from cryptography.fernet import Fernet

DB = 'claims.db'
if os.path.exists(DB):
    os.remove(DB)

def utcnow():
    return datetime.datetime.now(datetime.timezone.utc).strftime(
        '%Y-%m-%dT%H:%M:%S.%f')[:-3] + 'Z'

print('Lab ready. Fresh database:', DB)

Lab ready. Fresh database: claims.db


---
# Part A - Schema, roles and the access matrix

### Write the matrix before you write any policy

Every ambiguity you leave in this table becomes a vulnerability in the SQL.

| Role | Rows visible | Can modify | Never sees |
|---|---|---|---|
| **Claimant** | Own claims only | Contact details; may withdraw | Other claimants, adjuster notes, fraud score |
| **Adjuster** | Claims assigned to them | Status, notes, payout | NI number, sort code & account, other queues |
| **Fraud analyst** | Flagged claims, all adjusters | Fraud score, investigation notes | Bank account, claimant contact |
| **Auditor** | All claims, read-only | Nothing | Decrypted PII - sees ciphertext only |
| **ClaimMitra agent** | Claims in the active conversation | Status transitions within policy | NI number, bank details, fraud score |

### One design decision up front

`nino` is declared **BLOB**, not TEXT, from the very first migration.
Retrofitting a column type onto a live table is the expensive way to learn
this lesson - and the column has to hold ciphertext by Part D.

In [5]:
con = sqlite3.connect(DB, timeout=10)
con.row_factory = sqlite3.Row
# WAL lets the audit connection in Part E write while this one reads.
# Without it, the two connections deadlock - which is exactly the problem a
# separate audit store solves properly in production.
con.execute('PRAGMA journal_mode=WAL')
cur = con.cursor()

cur.executescript('''
CREATE TABLE users (
    user_id    INTEGER PRIMARY KEY,
    username   TEXT    NOT NULL UNIQUE,
    role       TEXT    NOT NULL CHECK (role IN
                 ('claimant','adjuster','fraud_analyst','auditor','agent')),
    full_name  TEXT    NOT NULL
);

CREATE TABLE claims (
    claim_id     TEXT    PRIMARY KEY,
    claimant_id  INTEGER NOT NULL REFERENCES users(user_id),
    handler_id   INTEGER          REFERENCES users(user_id),
    status       TEXT    NOT NULL CHECK (status IN
                   ('submitted','under_review','approved','rejected','withdrawn')),
    amount       REAL    NOT NULL,
    nino         BLOB,            -- National Insurance no., ciphertext from Part D
    nino_bidx    TEXT,            -- blind index for exact-match lookup
    sort_code    BLOB,            -- UK sort code, encrypted
    account_no   BLOB,            -- UK account number, encrypted
    contact      TEXT,
    fraud_flag   INTEGER NOT NULL DEFAULT 0,
    fraud_score  REAL,
    description  TEXT,
    filed_on     TEXT    NOT NULL
);

CREATE TABLE assessments (
    assessment_id INTEGER PRIMARY KEY AUTOINCREMENT,
    claim_id      TEXT NOT NULL REFERENCES claims(claim_id),
    adjuster_id   INTEGER NOT NULL REFERENCES users(user_id),
    notes         TEXT,
    created_on    TEXT NOT NULL
);

-- Layer 4. Append-only by convention here; in production this lives in a
-- separate store with INSERT-only grants and a different blast radius.
CREATE TABLE audit_log (
    event_id      INTEGER PRIMARY KEY AUTOINCREMENT,
    timestamp_utc TEXT NOT NULL,
    actor_id      INTEGER,
    on_behalf_of  INTEGER,
    action        TEXT NOT NULL,
    resource      TEXT NOT NULL,
    before_hash   TEXT,
    after_hash    TEXT,
    source_ip     TEXT,
    user_agent    TEXT,
    session_id    TEXT,
    result        TEXT NOT NULL,
    reason        TEXT,
    prev_hash     TEXT NOT NULL,
    row_hash      TEXT NOT NULL
);
''')
con.commit()
print('Schema created: users, claims, assessments, audit_log')

Schema created: users, claims, assessments, audit_log


In [6]:
users = [
    (1, 'r.whitfield', 'claimant',      'Rhea Whitfield'),
    (2, 'a.okonkwo',   'claimant',      'Adam Okonkwo'),
    (3, 'd.patel',     'claimant',      'Divya Patel'),
    (10, 'k.docherty', 'adjuster',      'Kirsty Docherty'),
    (11, 's.ellery',   'adjuster',      'Simon Ellery'),
    (20, 'f.khan',     'fraud_analyst', 'Farah Khan'),
    (30, 'audit.svc',  'auditor',       'Internal Audit'),
    (40, 'claimmitra', 'agent',         'ClaimMitra Agent'),
]
cur.executemany('INSERT INTO users VALUES (?,?,?,?)', users)

claims = [
    # claim_id, claimant, handler, status, amount(GBP), NI number, sort code,
    # account no, contact, fraud_flag, fraud_score, description, filed_on
    ('CLM-001', 1, 10, 'under_review', 4850.0, 'QQ123456C', '20-45-77', '31926544',
     'rhea@example.co.uk', 0, 0.08,
     'Rear-end collision on the A1(M), bumper and boot damage', '2026-09-14'),
    ('CLM-002', 1, 10, 'submitted',    1230.0, 'QQ123456C', '20-45-77', '31926544',
     'rhea@example.co.uk', 0, 0.03,
     'Windscreen cracked by road debris on the M6', '2026-10-02'),
    ('CLM-003', 2, 11, 'approved',    21000.0, 'AB654321D', '30-96-12', '84157203',
     'adam@example.co.uk', 0, 0.11,
     'Flood damage to vehicle, total loss assessment requested', '2026-08-30'),
    ('CLM-004', 2, 10, 'under_review', 6700.0, 'AB654321D', '30-96-12', '84157203',
     'adam@example.co.uk', 1, 0.78,
     'Third claim this year, same garage. My NI number is AB654321D', '2026-10-11'),
    ('CLM-005', 3, 11, 'submitted',     940.0, 'JT998877A', '60-11-09', '47720318',
     'divya@example.co.uk', 0, 0.05,
     'Side mirror replacement after parking incident, Leeds LS1 4DY', '2026-10-18'),
    ('CLM-006', 3, 11, 'rejected',     3100.0, 'JT998877A', '60-11-09', '47720318',
     'divya@example.co.uk', 0, 0.22,
     'Claim filed outside policy window', '2026-07-05'),
    ('CLM-007', 2, 11, 'under_review',15500.0, 'AB654321D', '30-96-12', '84157203',
     'adam@example.co.uk', 1, 0.91,
     'Engine seizure, pre-existing damage suspected', '2026-10-21'),
]
cur.executemany(
    'INSERT INTO claims (claim_id, claimant_id, handler_id, status, amount, '
    'nino, sort_code, account_no, contact, fraud_flag, fraud_score, '
    'description, filed_on) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?)', claims)

cur.executemany(
    'INSERT INTO assessments (claim_id, adjuster_id, notes, created_on) '
    'VALUES (?,?,?,?)',
    [('CLM-001', 10, 'Garage estimate received, awaiting surveyor', '2026-09-16'),
     ('CLM-004', 10, 'Pattern flagged - escalate to fraud desk', '2026-10-12'),
     ('CLM-007', 11, 'Requesting full service history before assessment', '2026-10-22')])
con.commit()

print(f'Seeded {len(users)} users and {len(claims)} claims.')
for r in cur.execute('SELECT role, COUNT(*) n FROM users GROUP BY role'):
    print(f'  {r["role"]:<14} {r["n"]}')

Seeded 8 users and 7 claims.
  adjuster       2
  agent          1
  auditor        1
  claimant       3
  fraud_analyst  1


---
# Part B - Row-Level Security via filtered views

### The concept

> *"The database enforces what the application was supposed to be doing all along."*

Without RLS, every endpoint, every query, every developer, forever must
remember the `WHERE claimant_id = ?` filter. One miss exposes everything.
With RLS the filter is no longer optional.

### SQLite has no native RLS

SQLite was designed as an embedded, single-application store - D. Richard Hipp
wrote it in 2000 aboard a US Navy destroyer so damage-control software would
keep working when the database server was unreachable. Single-user by design is
a feature, not an oversight.

So we retrofit enforcement with **a session context table plus filtered views**:

1. A `TEMP TABLE session_ctx` holds who is asking. TEMP is the security
   boundary - it is private to one connection and vanishes when it closes, so
   one request cannot read or poison another's context.
2. One **TEMP VIEW** per role. The view *is* the policy.
3. The application never touches the base table.

> **Why TEMP VIEW and not CREATE VIEW?** A permanent view in the main database
> cannot reference a temp object - SQLite rejects it. Making the view temp as
> well is both required and more honest: the view is session-scoped, exactly
> like the identity it depends on.

In [7]:
def set_session(user_id):
    """Establish WHO is asking, for this connection only.

    In production this is set in the connection-pool checkout hook from the
    verified server-side session - NEVER from a client-supplied value."""
    cur.execute('DELETE FROM session_ctx')
    cur.execute(
        'INSERT INTO session_ctx (user_id, role, session_id, source_ip) '
        'SELECT user_id, role, ?, ? FROM users WHERE user_id = ?',
        (f'sess-{user_id}-abc123', '10.4.22.17', user_id))
    con.commit()


cur.executescript('''
CREATE TEMP TABLE session_ctx (
    user_id    INTEGER,
    role       TEXT,
    session_id TEXT,
    source_ip  TEXT
);

-- Claimants: own claims only. Note what is ABSENT from the column list -
-- fraud_score and handler notes simply do not exist for this role.
CREATE TEMP VIEW claimant_view AS
  SELECT claim_id, claimant_id, status, amount, contact, description, filed_on
  FROM   claims
  WHERE  claimant_id = (SELECT user_id FROM session_ctx);

-- Adjusters: assigned claims only. No NI number, no bank details.
CREATE TEMP VIEW adjuster_view AS
  SELECT claim_id, claimant_id, handler_id, status, amount,
         description, fraud_flag, filed_on
  FROM   claims
  WHERE  handler_id = (SELECT user_id FROM session_ctx);

-- Fraud analysts: flagged claims across ALL adjusters.
CREATE TEMP VIEW fraud_view AS
  SELECT claim_id, claimant_id, handler_id, status, amount,
         fraud_flag, fraud_score, description
  FROM   claims
  WHERE  fraud_flag = 1;

-- Auditors: everything, read-only. nino is included deliberately -
-- after Part D they will see ciphertext and nothing more.
CREATE TEMP VIEW auditor_view AS
  SELECT claim_id, claimant_id, handler_id, status, amount, nino, filed_on
  FROM   claims;
''')
con.commit()
print('session_ctx + 4 role views created.')

session_ctx + 4 role views created.


In [8]:
# Prove the isolation. Rhea (user 1) owns CLM-001 and CLM-002.
set_session(1)
rows = cur.execute('SELECT claim_id, status, amount FROM claimant_view').fetchall()
print('Rhea Whitfield (claimant, user 1) sees:')
for r in rows:
    print(f'  {r["claim_id"]}  {r["status"]:<13} £{r["amount"]:>10,.0f}')

print()
# Adam (user 2) owns CLM-003, CLM-004, CLM-007 - including the Rs 155,000 one.
set_session(2)
rows = cur.execute('SELECT claim_id, status, amount FROM claimant_view').fetchall()
print('Adam Okonkwo (claimant, user 2) sees:')
for r in rows:
    print(f'  {r["claim_id"]}  {r["status"]:<13} £{r["amount"]:>10,.0f}')

print()
print('Same query text. Different rows. The filter is in the engine, not the app.')

Rhea Whitfield (claimant, user 1) sees:
  CLM-001  under_review  £     4,850
  CLM-002  submitted     £     1,230

Adam Okonkwo (claimant, user 2) sees:
  CLM-003  approved      £    21,000
  CLM-004  under_review  £     6,700
  CLM-007  under_review  £    15,500

Same query text. Different rows. The filter is in the engine, not the app.


In [9]:
# The lesson that matters most in this notebook.
set_session(1)                      # Rhea again

via_view = cur.execute('SELECT COUNT(*) c FROM claimant_view').fetchone()['c']
via_base = cur.execute('SELECT COUNT(*) c FROM claims').fetchone()['c']

print(f'Rhea via claimant_view : {via_view} claims')
print(f'Rhea via base table    : {via_base} claims   <-- ALL OF THEM')
print()
leak = cur.execute(
    'SELECT claim_id, claimant_id, amount, fraud_score FROM claims '
    'WHERE claimant_id != 1 LIMIT 3').fetchall()
print('What she can reach by querying `claims` directly:')
for r in leak:
    print(f'  {r["claim_id"]}  owner={r["claimant_id"]}  '
          f'£{r["amount"]:>10,.0f}  fraud_score={r["fraud_score"]}')
print()
print('This is enforcement by CONVENTION plus code review, not by the engine.')
print('Grant no base-table access, and treat the DB file itself as the real')
print('boundary. If the data matters, migrate to PostgreSQL before production.')

Rhea via claimant_view : 2 claims
Rhea via base table    : 7 claims   <-- ALL OF THEM

What she can reach by querying `claims` directly:
  CLM-003  owner=2  £    21,000  fraud_score=0.11
  CLM-004  owner=2  £     6,700  fraud_score=0.78
  CLM-005  owner=3  £       940  fraud_score=0.05

This is enforcement by CONVENTION plus code review, not by the engine.
Grant no base-table access, and treat the DB file itself as the real
boundary. If the data matters, migrate to PostgreSQL before production.


### What PostgreSQL would do instead

```sql
ALTER TABLE claims ENABLE ROW LEVEL SECURITY;
ALTER TABLE claims FORCE  ROW LEVEL SECURITY;   -- the step most teams miss

CREATE POLICY claimant_rows ON claims
  FOR ALL TO claimmitra_app
  USING      (claimant_id = current_setting('app.user_id')::int)
  WITH CHECK (claimant_id = current_setting('app.user_id')::int);
```

| Clause | Purpose |
|---|---|
| `USING` | Filters rows you can **see** |
| `WITH CHECK` | Validates rows you **write**. Omit it and a user can update a row into someone else's ownership, then lose sight of it |
| `FORCE` | Applies policies to the table owner too. Without it, if your migration user owns the table and the app reuses that connection, RLS is decoration |

### Five ways RLS fails in production

| Failure | What actually happens |
|---|---|
| App connects as superuser/table owner | Every policy silently skipped. Looks secured, enforces nothing |
| Policy on `SELECT` only | Reads scoped, `UPDATE`/`DELETE` wide open. Cannot see other rows, can still destroy them |
| JOIN leakage | `claims` protected, `claim_documents` not. One JOIN and the data walks out through the child table |
| Session variable never set | Predicate turns false, queries return zero rows. The usual "fix" is disabling RLS |
| **Connection pool reuse** | A pooled connection keeps the previous request's identity. User B reads user A's rows |

The last one is the dangerous one: it passes every test suite, because tests
use fresh connections. It only leaks in production, under load.

In [10]:
# Demonstrate the connection-pool bug - the one that passes every test.
COUNT_SQL = 'SELECT COUNT(*) c FROM claimant_view'

print('Request 1: Adam (user 2) checks out a connection')
set_session(2)
print(f'  sees {cur.execute(COUNT_SQL).fetchone()["c"]} claims')

print()
print('Request 2: Rhea (user 1) gets the SAME pooled connection back,')
print('           and the checkout hook forgot to reset the identity...')
# (deliberately NOT calling set_session(1))
rows = cur.execute('SELECT claim_id, amount FROM claimant_view').fetchall()
print(f'  Rhea now sees {len(rows)} claims - and they are ARJUN\'S:')
for r in rows:
    print(f'    {r["claim_id"]}  £{r["amount"]:>10,.0f}')
print()
print('Fix: RESET the context on checkin, or set identity inside the')
print('transaction (SET LOCAL in Postgres). Fail closed on a missing identity.')

Request 1: Adam (user 2) checks out a connection
  sees 3 claims

Request 2: Rhea (user 1) gets the SAME pooled connection back,
           and the checkout hook forgot to reset the identity...
  Rhea now sees 3 claims - and they are ARJUN'S:
    CLM-003  £    21,000
    CLM-004  £     6,700
    CLM-007  £    15,500

Fix: RESET the context on checkin, or set identity inside the
transaction (SET LOCAL in Postgres). Fail closed on a missing identity.


---
# Part C - Write enforcement with INSTEAD OF triggers

A view gives you **read** security. Views are not writable, so without more
work your application writes straight to the base table and bypasses every
policy you just built. `INSTEAD OF` triggers close that gap.

### Three guards, and why each exists

1. **Ownership re-check.** The view already filtered, but a trigger fires on
   whatever row reaches it. Re-verify - this is the confused-deputy defence.
2. **Allowed transitions.** Authorisation is not just *which rows*, but *which
   state changes*. "A claimant may withdraw, never approve" is a business rule
   the database can hold.
3. **Immutable fields.** Payout amount is adjuster territory. Encoding that
   here means no API bug can ever move it.

> `RAISE(ABORT)` rolls back the statement but keeps the surrounding transaction
> alive - distinct from `RAISE(ROLLBACK)`, which kills everything. Choosing the
> wrong one is a classic production bug: a rejected edit silently discards the
> audit entry written moments earlier in the same transaction.

In [11]:
cur.executescript('''
CREATE TEMP TRIGGER claimant_update
INSTEAD OF UPDATE ON claimant_view
BEGIN
  -- guard 1: ownership re-check, never trust the view alone
  SELECT RAISE(ABORT, 'DENIED: not your claim')
  WHERE  OLD.claimant_id <> (SELECT user_id FROM session_ctx);

  -- guard 2: a claimant may only withdraw, never approve or reject
  SELECT RAISE(ABORT, 'DENIED: status transition not permitted for claimant')
  WHERE  NEW.status <> OLD.status AND NEW.status <> 'withdrawn';

  -- guard 3: the payout amount is not claimant-editable
  SELECT RAISE(ABORT, 'DENIED: amount is immutable for this role')
  WHERE  NEW.amount <> OLD.amount;

  -- only now, the actual write
  UPDATE claims
  SET    status  = NEW.status,
         contact = NEW.contact
  WHERE  claim_id = OLD.claim_id;
END;

CREATE TEMP TRIGGER claimant_delete
INSTEAD OF DELETE ON claimant_view
BEGIN
  SELECT RAISE(ABORT, 'DENIED: claims are never deleted, only withdrawn');
END;
''')
con.commit()
print('INSTEAD OF triggers installed on claimant_view (UPDATE, DELETE).')

INSTEAD OF triggers installed on claimant_view (UPDATE, DELETE).


In [12]:
def attempt(label, sql, params=()):
    try:
        cur.execute(sql, params)
        con.commit()
        print(f'  ALLOWED  {label}')
        return True
    except sqlite3.IntegrityError as e:
        print(f'  BLOCKED  {label}')
        print(f'           -> {e}')
        return False


set_session(1)                                  # Rhea
print('Rhea Whitfield (claimant) attempts four writes:')
print()

attempt('update her own contact details',
        "UPDATE claimant_view SET contact='rhea.sharma@example.co.uk' "
        "WHERE claim_id='CLM-001'")

attempt('withdraw her own claim',
        "UPDATE claimant_view SET status='withdrawn' WHERE claim_id='CLM-002'")

attempt('APPROVE her own claim',
        "UPDATE claimant_view SET status='approved' WHERE claim_id='CLM-001'")

attempt('raise her own payout to £500,000',
        "UPDATE claimant_view SET amount=500000 WHERE claim_id='CLM-001'")

attempt('delete a claim outright',
        "DELETE FROM claimant_view WHERE claim_id='CLM-001'")

print()
for r in cur.execute(
        "SELECT claim_id, status, amount, contact FROM claims "
        "WHERE claimant_id=1 ORDER BY claim_id"):
    print(f'  {r["claim_id"]}  {r["status"]:<13} £{r["amount"]:>9,.0f}  {r["contact"]}')

Rhea Whitfield (claimant) attempts four writes:

  ALLOWED  update her own contact details
  ALLOWED  withdraw her own claim
  BLOCKED  APPROVE her own claim
           -> DENIED: status transition not permitted for claimant
  BLOCKED  raise her own payout to £500,000
           -> DENIED: amount is immutable for this role
  BLOCKED  delete a claim outright
           -> DENIED: claims are never deleted, only withdrawn

  CLM-001  under_review  £    4,850  rhea.sharma@example.co.uk
  CLM-002  withdrawn     £    1,230  rhea@example.co.uk


---
# Part D - Field-level encryption, envelope keys, blind index

### Where Layer 1 stops working

RLS answers *"which rows may this user query?"* It has nothing to say about
anyone who reads the bytes without going through a query at all:

| Attacker | RLS | Field encryption |
|---|---|---|
| The DBA | Not protected - sees plaintext in every query | Protected |
| A stolen `.db` file or snapshot | Not protected - no engine, no policies | Protected |
| The backup pipeline | Three bypasses in one process | Protected |
| A stray log line or APM trace | Not protected - nobody wrote a policy for Elasticsearch | Protected |

This is the Capital One lesson stated precisely. The attacker reached the S3
buckets with **valid credentials** - no TLS failure, no RLS bypass, nothing
"broken". Had the PII columns been encrypted with keys held outside S3, the
same intrusion would have yielded 106 million rows of unreadable ciphertext.

### Choosing an algorithm (and not inventing one)

| Scheme | Use for | Do not use for |
|---|---|---|
| **Fernet** (AES-128-CBC + HMAC-SHA256) | Application field encryption in Python. Safe defaults, very hard to misuse | Anything needing AES-256 or non-Python interop |
| **AES-256-GCM** | The production default. Hardware-accelerated, FIPS-approvable | Reusing a nonce. Ever - it leaks the key stream |
| **RSA-OAEP** | Encrypting other keys, and signatures | Bulk data - thousands of times slower |
| **ECB mode** | Nothing. Ever | Everything - identical plaintext yields identical ciphertext |

*Fernet is named after Fernet-Branca, the aggressively bitter Italian digestif
- the joke being that it is good for you and nobody wants to tamper with it
twice. ECB has its own folklore: the "ECB penguin", an image of Tux encrypted
in ECB mode where the penguin remains perfectly visible, retired ECB from
production more effectively than any academic paper.*

### Envelope encryption - because keys are the hard part

Encrypting data is easy. Managing keys is where real failures happen.

| Level | Where the key lives | Problem |
|---|---|---|
| 0 | Hardcoded in source | In Git forever, including every fork and laptop. Rotation means a deploy |
| 1 | Environment variable | Readable by any process on the host, visible in `/proc`, captured by crash dumps |
| 2 | Secret manager at boot | Audited and centrally rotated - but still in process memory for the lifetime |
| 3 | **KMS / HSM, envelope encryption** | Master key never leaves the HSM. A memory dump yields one data key, not the kingdom |

```
DEK = random key per record or tenant  ->  encrypts the data   (fast, local, disposable)
KEK = master key, lives in the HSM     ->  encrypts the DEK    (rotate it, no data re-encryption)
```

Rotating the KEK is instant. Re-encrypting a billion rows is not. That
asymmetry is the entire reason the pattern exists.

In [13]:
# ---- Level 3 simulated: a KEK that never leaves its boundary -----------
# In production: AWS KMS, Azure Key Vault or HashiCorp Vault. The KEK object
# here stands in for the HSM - note that we only ever ask it to wrap/unwrap,
# never to hand us the master key itself.

class KeyManagementService:
    """Stand-in for an HSM. The master key never leaves this object."""
    def __init__(self):
        self._kek = Fernet(Fernet.generate_key())
        self.unwrap_calls = 0

    def generate_data_key(self):
        """Returns (plaintext DEK, wrapped DEK). Store only the wrapped one."""
        dek = Fernet.generate_key()
        return dek, self._kek.encrypt(dek)

    def unwrap(self, wrapped_dek):
        self.unwrap_calls += 1
        return self._kek.decrypt(wrapped_dek)


kms = KeyManagementService()
dek_plain, dek_wrapped = kms.generate_data_key()

# What gets persisted alongside the data is the WRAPPED key.
with open('claims.dek', 'wb') as f:
    f.write(dek_wrapped)

print('KEK  : inside the KMS, never exported')
print(f'DEK  : wrapped and stored on disk ({len(dek_wrapped)} bytes)')
print(f'       {dek_wrapped[:48].decode()}...')
print()
print('An attacker who steals claims.db AND claims.dek still has nothing -')
print('the wrapped DEK is useless without a KMS call they cannot make.')

KEK  : inside the KMS, never exported
DEK  : wrapped and stored on disk (140 bytes)
       gAAAAABq4OidRaJvGG4OPB0Zyogp264JG23D8NV2RJ9uF7kh...

An attacker who steals claims.db AND claims.dek still has nothing -
the wrapped DEK is useless without a KMS call they cannot make.


In [14]:
# ---- Field encryption + blind index -----------------------------------
# An encrypted column cannot be indexed, sorted, or queried with LIKE.
# "Find the claim with NI number QQ123456C" stops working. A blind index -
# HMAC(nino) stored alongside - restores EXACT-MATCH lookup without revealing
# the value. Range queries are simply gone; plan for that.

BIDX_KEY = os.urandom(32)          # in production: a second key from the KMS

def load_cipher():
    with open('claims.dek', 'rb') as f:
        return Fernet(kms.unwrap(f.read()))

def encrypt_field(plaintext):
    return load_cipher().encrypt(plaintext.encode()) if plaintext else None

def decrypt_field(ciphertext):
    return load_cipher().decrypt(ciphertext).decode() if ciphertext else None

def blind_index(plaintext):
    """Deterministic keyed hash - equal inputs give equal output, so exact
    match works; without the key the value cannot be recovered or guessed."""
    return hmac.new(BIDX_KEY, plaintext.encode(), hashlib.sha256).hexdigest()


# Migrate NI number, sort code and account number to ciphertext.
rows = cur.execute(
    'SELECT claim_id, nino, sort_code, account_no FROM claims '
    'WHERE nino IS NOT NULL').fetchall()

migrated = 0
for r in rows:
    def _plain(v):
        return v.decode() if isinstance(v, bytes) else v
    nino = _plain(r['nino'])
    cur.execute(
        'UPDATE claims SET nino=?, nino_bidx=?, sort_code=?, account_no=? '
        'WHERE claim_id=?',
        (encrypt_field(nino), blind_index(nino),
         encrypt_field(_plain(r['sort_code'])),
         encrypt_field(_plain(r['account_no'])), r['claim_id']))
    migrated += 1
con.commit()
print(f'Migrated {migrated} NI numbers to ciphertext + blind index.')
print(f'KMS unwrap calls so far: {kms.unwrap_calls}')

Migrated 7 NI numbers to ciphertext + blind index.
KMS unwrap calls so far: 21


In [15]:
# What the auditor sees now.
set_session(30)
print('Auditor queries auditor_view:')
print()
for r in cur.execute('SELECT claim_id, nino FROM auditor_view LIMIT 4'):
    blob = r['nino']
    print(f'  {r["claim_id"]}  {blob[:52].decode()}...  ({len(blob)} bytes)')

print()
print('Same bytes, decrypted by a service holding the DEK:')
for r in cur.execute('SELECT claim_id, nino FROM claims LIMIT 4'):
    print(f'  {r["claim_id"]}  {decrypt_field(r["nino"])}')

print()
print('Exact-match lookup still works, via the blind index:')
target = 'AB654321D'
hits = cur.execute('SELECT claim_id, amount FROM claims WHERE nino_bidx=?',
                   (blind_index(target),)).fetchall()
print(f'  WHERE nino_bidx = HMAC("{target}")  ->  {len(hits)} claims')
for h in hits:
    print(f'    {h["claim_id"]}  £{h["amount"]:>10,.0f}')
print()
print('Note: no LIKE, no ORDER BY, no range query on this column. That is')
print('the real cost of field-level encryption - design for it up front.')

Auditor queries auditor_view:

  CLM-001  gAAAAABq4OidTbgjUmhsDNft5uyZncfyyKsFwGjua9vBaZO85-1w...  (100 bytes)
  CLM-002  gAAAAABq4OiduLPu5cFsJn-9WlXDv0NP3IOhSnFX0hOIxQEPIaOo...  (100 bytes)
  CLM-003  gAAAAABq4Oid9F9XarQK-FpxVPiE3x8KyoLMAiycTbpP6LXu8nTf...  (100 bytes)
  CLM-004  gAAAAABq4OidLOPCAw72a4wjv8xF8Xi7jENJf_T464FijHPscIAm...  (100 bytes)

Same bytes, decrypted by a service holding the DEK:
  CLM-001  QQ123456C
  CLM-002  QQ123456C
  CLM-003  AB654321D
  CLM-004  AB654321D

Exact-match lookup still works, via the blind index:
  WHERE nino_bidx = HMAC("AB654321D")  ->  3 claims
    CLM-003  £    21,000
    CLM-004  £     6,700
    CLM-007  £    15,500

Note: no LIKE, no ORDER BY, no range query on this column. That is
the real cost of field-level encryption - design for it up front.


---
# Part E - Tamper-evident audit logging

Layers 1-3 **prevent**. Layer 4 **proves**. Regulators do not ask whether you
protected the data - they ask you to demonstrate who touched what, when, and
whether the record can be trusted.

*Tamper-evident logging comes to computing from aviation. The 1956 Grand Canyon
mid-air collision killed 128 people and left investigators with no record of
the final minutes; the resulting mandate produced the crash-survivable flight
recorder. The principle transferred intact: a record of critical actions is
worthless unless it survives the party who would most like to change it.*

### The nine mandatory fields

| Field | Why it exists | The mistake teams make |
|---|---|---|
| `event_id` | Unique, never reused - the join key for the investigation | Auto-increment reveals volume and makes gaps invisible |
| `timestamp_utc` | UTC, microsecond precision, synchronised source | Local time zones - correlation becomes guesswork |
| `actor_id` + `on_behalf_of` | Who acted, and whose authority they used | Logging only the service account for agent access |
| `action` | read / write / delete / export / login / **denied** | Recording only successes - denials are the attack signal |
| `resource` | Table plus row id, and which columns | "accessed claims" with no row id - useless for scoping |
| `before_hash` / `after_hash` | Proves what changed without storing the value again | Storing old and new plaintext - a second PII store |
| `source_ip` + `user_agent` + `session_id` | Where from, which session - ties events into one story | Capturing the load balancer IP instead of the client |
| `result` + `reason` | success / failure / denied, and the deciding policy | No reason field, so a denial looks like a bug |
| `row_hash` | SHA-256 over previous hash and this record | Omitted - the log is editable, and so inadmissible |

### The agent changes the math

A human adjuster opens perhaps sixty claims a day and anomalies are legible.
ClaimMitra issues thousands of queries an hour, across every claim in its
scope, on behalf of many different users, through **one service identity**.
An audit record that only says `claimmitra-svc` is forensically useless -
hence `on_behalf_of`.

In [16]:
GENESIS = '0' * 64

# Audit writes go through their OWN connection. Two reasons: it avoids nested
# transaction problems with the triggers above, and it mirrors production,
# where the audit store is a separate system with different credentials and a
# different blast radius.
audit_con = sqlite3.connect(DB, timeout=10)
audit_con.row_factory = sqlite3.Row
audit_cur = audit_con.cursor()


def canonical(record):
    """Deterministic serialisation. Sorted keys and fixed separators, or a
    harmless re-serialisation later looks exactly like tampering."""
    return json.dumps(record, sort_keys=True, separators=(',', ':'))


def chain_hash(prev_hash, record):
    return hashlib.sha256((prev_hash + canonical(record)).encode()).hexdigest()


def last_hash():
    row = audit_cur.execute(
        'SELECT row_hash FROM audit_log ORDER BY event_id DESC LIMIT 1').fetchone()
    return row['row_hash'] if row else GENESIS


def write_audit(actor_id, action, resource, result,
                on_behalf_of=None, before=None, after=None,
                reason=None, source_ip='10.4.22.17',
                user_agent='ClaimMitra/2.1', session_id=None):
    """Append one tamper-evident audit record."""
    prev = last_hash()
    record = {
        'timestamp_utc': utcnow(),
        'actor_id':      actor_id,
        'on_behalf_of':  on_behalf_of,
        'action':        action,
        'resource':      resource,
        'before_hash':   hashlib.sha256(str(before).encode()).hexdigest() if before else None,
        'after_hash':    hashlib.sha256(str(after).encode()).hexdigest() if after else None,
        'source_ip':     source_ip,
        'user_agent':    user_agent,
        'session_id':    session_id,
        'result':        result,
        'reason':        reason,
    }
    rh = chain_hash(prev, record)
    audit_cur.execute(
        'INSERT INTO audit_log (timestamp_utc, actor_id, on_behalf_of, action, '
        'resource, before_hash, after_hash, source_ip, user_agent, session_id, '
        'result, reason, prev_hash, row_hash) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?)',
        (record['timestamp_utc'], record['actor_id'], record['on_behalf_of'],
         record['action'], record['resource'], record['before_hash'],
         record['after_hash'], record['source_ip'], record['user_agent'],
         record['session_id'], record['result'], record['reason'], prev, rh))
    audit_con.commit()
    return rh


print('Audit writer ready (separate connection).')

Audit writer ready (separate connection).


In [17]:
# A realistic sequence: human access, agent access on behalf of a user,
# a denied attempt, and an export. Note that denials are logged too.

write_audit(10, 'read', 'claims/CLM-001', 'success',
            session_id='sess-10-9f2a')

write_audit(10, 'update', 'claims/CLM-001#status', 'success',
            before='under_review', after='approved',
            session_id='sess-10-9f2a', reason='surveyor report received')

# ClaimMitra acting FOR Rhea - both identities recorded.
write_audit(40, 'read', 'claims/CLM-002', 'success',
            on_behalf_of=1, session_id='sess-40-cc71',
            user_agent='ClaimMitra/2.1 (agent)')

# The attack signal: a denied attempt.
write_audit(1, 'update', 'claims/CLM-001#status', 'denied',
            reason='policy: claimant may not approve',
            session_id='sess-1-44bd', user_agent='Mozilla/5.0')

write_audit(1, 'update', 'claims/CLM-001#amount', 'denied',
            reason='policy: amount immutable for role claimant',
            session_id='sess-1-44bd', user_agent='Mozilla/5.0')

write_audit(30, 'export', 'claims/*', 'success',
            reason='quarterly internal audit', session_id='sess-30-e001',
            user_agent='audit-runner/1.4')

n = audit_cur.execute('SELECT COUNT(*) c FROM audit_log').fetchone()['c']
print(f'{n} audit records written.')
print()
print(f'{"id":>3}  {"actor":>5} {"obo":>4}  {"action":<8}{"resource":<26}'
      f'{"result":<9}hash')
print('-' * 86)
for r in audit_cur.execute('SELECT * FROM audit_log ORDER BY event_id'):
    obo = r['on_behalf_of'] if r['on_behalf_of'] is not None else '-'
    print(f'{r["event_id"]:>3}  {r["actor_id"]:>5} {str(obo):>4}  '
          f'{r["action"]:<8}{r["resource"]:<26}{r["result"]:<9}'
          f'{r["row_hash"][:12]}...')

6 audit records written.

 id  actor  obo  action  resource                  result   hash
--------------------------------------------------------------------------------------
  1     10    -  read    claims/CLM-001            success  460e6b0e3a17...
  2     10    -  update  claims/CLM-001#status     success  0adfe71e3de1...
  3     40    1  read    claims/CLM-002            success  256eef2b1e21...
  4      1    -  update  claims/CLM-001#status     denied   5b526d146000...
  5      1    -  update  claims/CLM-001#amount     denied   4579595a986d...
  6     30    -  export  claims/*                  success  66f996ada88c...


### Six audit-logging failures

| Failure | The fix |
|---|---|
| PII in the log body | Store hashes and references. Never a raw NI number, never a full request body |
| Audit table in the application database | Separate store, separate credentials, append-only grants |
| Logging only successes | Log denials with the deciding policy. Alert on denial *rate* |
| No retention policy | Hot 90 days queryable, cold 7 years in object storage, defined up front |
| Restore never tested | Quarterly drill: pull a random day, verify the chain, query it |
| Clock drift | NTP everywhere, monotonic sequence numbers alongside timestamps |

Note that our `before_hash`/`after_hash` columns hold **hashes, not values** -
that is the first row of this table, implemented. We can prove `under_review`
became `approved` without ever creating a second copy of the data.

---
# Part F - Agent scoping, PII scanning, and UK/EU compliance

### The agent is a principal, not a superuser

ClaimMitra gets its own narrow role, scoped to the claims in the conversation
it is actually having - not to everything its user could theoretically reach.
An agent inherits the blast radius of its credentials, so make that radius
small.

### Four ways agents leak data at rest

All four are stores that were **never classified as stores**. The data
protection review covered "the database" and missed every one of them.

| Store | Why it is invisible | Control |
|---|---|---|
| **Vector store** | RAG embeddings derive from documents that held PII. Provisioned as "infrastructure" - no RLS, no encryption, no retention | Classify at the sensitivity of the source corpus. Partition per tenant. Include in erasure requests |
| **Conversation memory** | Agent checkpoints hold whatever the user said, including the sort code they pasted. No data-flow diagram shows it | Encrypt checkpoint state, same retention clock, scan on the way in |
| **Tool arguments in traces** | `get_claim(pan="ABCDE1234F")` is now in your APM vendor, replicated and retained for a year | Scrub at the exporter, not the dashboard. Allowlist recordable arguments |
| **Fine-tuning corpora** | Consent for claim processing is not consent for model training, and erasure cannot reach a trained weight | Anonymise before the corpus is built. Keep lineage to the lawful basis |

In [18]:
# Scope ClaimMitra to one conversation, then prove the boundary holds.
cur.execute('CREATE TEMP TABLE IF NOT EXISTS agent_scope (claim_id TEXT)')

def open_agent_session(agent_id, on_behalf_of, claim_ids):
    set_session(agent_id)
    cur.execute('DELETE FROM agent_scope')
    cur.executemany('INSERT INTO agent_scope VALUES (?)',
                    [(c,) for c in claim_ids])
    con.commit()
    write_audit(agent_id, 'session.open', f'claims/{",".join(claim_ids)}',
                'success', on_behalf_of=on_behalf_of,
                session_id=f'sess-{agent_id}-cc71',
                user_agent='ClaimMitra/2.1 (agent)')

cur.executescript('''
CREATE TEMP VIEW IF NOT EXISTS agent_view AS
  SELECT claim_id, claimant_id, status, amount, contact, filed_on
  FROM   claims
  WHERE  claim_id IN (SELECT claim_id FROM agent_scope);
''')
con.commit()

# Rhea opens a chat about CLM-002 only.
open_agent_session(40, on_behalf_of=1, claim_ids=['CLM-002'])

print('ClaimMitra session for Rhea, scoped to CLM-002:')
for r in cur.execute('SELECT claim_id, status, amount FROM agent_view'):
    print(f'  {r["claim_id"]}  {r["status"]:<12} £{r["amount"]:>9,.0f}')

agent_rows = cur.execute('SELECT COUNT(*) c FROM agent_view').fetchone()['c']
set_session(1)
rhea_rows = cur.execute('SELECT COUNT(*) c FROM claimant_view').fetchone()['c']

print()
print(f'  Rows reachable by the agent  : {agent_rows}')
print(f'  Rows Rhea could reach herself: {rhea_rows}')
print()
print('The agent is narrower than its own user. If the agent is compromised')
print('by prompt injection, the blast radius is one claim - not her whole file.')

ClaimMitra session for Rhea, scoped to CLM-002:
  CLM-002  withdrawn    £    1,230

  Rows reachable by the agent  : 1
  Rows Rhea could reach herself: 2

The agent is narrower than its own user. If the agent is compromised
by prompt injection, the blast radius is one claim - not her whole file.


In [19]:
# Presidio-style PII detection. Every control so far protects the columns you
# DECLARED sensitive. Agents generate free text, and customers paste NI numbers
# into all of it. CLM-004's description has one - look back at the seed data.

PATTERNS = {
    # UK National Insurance number: 2 prefix letters, 6 digits, 1 suffix A-D
    'UK_NINO':      re.compile(r'\b[A-CEGHJ-PR-TW-Z]{2}\s?\d{2}\s?\d{2}\s?\d{2}\s?[A-D]\b'),
    'UK_SORT_CODE': re.compile(r'\b\d{2}-\d{2}-\d{2}\b'),
    'UK_ACCOUNT':   re.compile(r'\b\d{8}\b'),
    'UK_POSTCODE':  re.compile(r'\b[A-Z]{1,2}\d[A-Z\d]?\s?\d[A-Z]{2}\b'),
    'EMAIL':        re.compile(r'\b[\w.+-]+@[\w-]+\.[\w.]+\b'),
    'UK_PHONE':     re.compile(r'\b(?:\+44\s?|0)7\d{3}\s?\d{6}\b'),
}

def scan(text):
    found = []
    for label, rx in PATTERNS.items():
        for m in rx.finditer(text or ''):
            found.append((label, m.group(), m.start(), m.end()))
    return found

def anonymise(text):
    out = text
    for label, rx in PATTERNS.items():
        out = rx.sub(f'<{label}>', out)
    return out


print('Scanning free-text claim descriptions:')
print()
hits = 0
# fetchall() first: we write to the audit connection inside this loop, and an
# open cursor on the other connection would hold a read lock across it.
for r in cur.execute('SELECT claim_id, description FROM claims').fetchall():
    found = scan(r['description'])
    if found:
        hits += 1
        print(f'  {r["claim_id"]}  FOUND {[(f[0], f[1]) for f in found]}')
        print(f'    before: {r["description"]}')
        print(f'    after : {anonymise(r["description"])}')
        write_audit(40, 'pii.detected', f'claims/{r["claim_id"]}#description',
                    'success', reason=f'{found[0][0]} in free text')

print()
print(f'{hits} of 7 descriptions contained an undeclared identifier.')
print()
print('Detection is probabilistic - it will miss an unusual format and will')
print('occasionally flag a product code. Treat it as a SAFETY NET: structured')
print('fields stay validated and encrypted explicitly; this catches what leaks')
print('into free text nobody designed for. Replacing explicit field handling')
print('with a scanner swaps a certainty for a probability.')

Scanning free-text claim descriptions:

  CLM-004  FOUND [('UK_NINO', 'AB654321D')]
    before: Third claim this year, same garage. My NI number is AB654321D
    after : Third claim this year, same garage. My NI number is <UK_NINO>
  CLM-005  FOUND [('UK_POSTCODE', 'LS1 4DY')]
    before: Side mirror replacement after parking incident, Leeds LS1 4DY
    after : Side mirror replacement after parking incident, Leeds <UK_POSTCODE>

2 of 7 descriptions contained an undeclared identifier.

Detection is probabilistic - it will miss an unusual format and will
occasionally flag a product code. Treat it as a SAFETY NET: structured
fields stay validated and encrypted explicitly; this catches what leaks
into free text nobody designed for. Replacing explicit field handling
with a scanner swaps a certainty for a probability.


### UK GDPR & DPA 2018 - five obligations with teeth

Lloyds is a UK institution, so the binding regime is **UK GDPR plus the Data
Protection Act 2018**, enforced by the **ICO**. The EU AI Act matters only
where output is used in the EU - scope that deliberately rather than assuming
it either way.

| Article | Obligation | What it means for ClaimMitra |
|---|---|---|
| **5(1)(b)** Purpose limitation | Specified, explicit, legitimate purposes; no incompatible further processing | May read an NI number to settle a claim. May **not** reuse it to pre-fill a marketing offer - same customer, same table, different purpose |
| **5(1)(e) + 17** Storage limitation, erasure | Kept no longer than necessary; erased on valid request | A settled 2019 claim still holding a plaintext sort code is a standing violation. Erasure must reach backups and the vector store |
| **5(1)(f) + 32** Security of processing | Appropriate measures - encryption and pseudonymisation named explicitly | The exact pair the ICO cited against Marriott. Parts B-E are the evidence |
| **30** Records of processing | Maintain records, available to the ICO on request | Your audit log is a statutory artefact, not only a detection control |
| **33 / 34** Breach notification | Notify the ICO within 72 h of becoming **aware**, where there is risk to rights | The clock starts at awareness. Marriott ran ~4 years before awareness |

**ICO fining tiers** (published maxima): standard **£8.7M or 2%** of worldwide
turnover; higher **£17.5M or 4%**, whichever is greater. Note which is which -
an Article 32 security failure sits in the *standard* tier, while breaching the
Article 5 principles sits in the *higher* one. Teams routinely assume the
opposite.

### EU AI Act - is ClaimMitra high-risk?

Resist the reflex to answer "it is AI, so it is high-risk". That is wrong, and
expensive in both directions.

Annex III point **5(c)** names *risk assessment and pricing in relation to
natural persons in the case of **life and health** insurance*. Point **5(b)**
names creditworthiness evaluation. General insurance claims handling is not
listed, and fraud detection is not listed under 5(c). But classification
follows **actual function**, not the product label - if the agent prices life
or health risk, it is in scope regardless of what the team calls it.

Where it *is* high-risk, **Article 12** is the one that lands on this session:

> "High-risk AI systems shall technically allow for the automatic recording of
> events (logs) over the lifetime of the system."

Part E is not merely good practice for such a system. It is the statutory
requirement, stated almost in our own terms. Administrative fines reach
EUR 35M or 7% of worldwide turnover for prohibited practices, and EUR 15M or
3% for most other operator obligations - a higher ceiling than GDPR.

In [20]:
# Map each obligation to the control that satisfies it, and show the evidence.
# This table is what you hand the ICO.

controls = [
    ('Art 5(1)(b) Purpose limitation', 'Role views expose only columns needed per purpose',
     'claimant_view has no fraud_score column'),
    ('Art 5(1)(e) Storage limitation', 'Withdrawal is a claimant-permitted transition',
     None),
    ('Art 5(1)(f) + 32 Security', 'RLS via session views + INSTEAD OF triggers',
     '3 of 5 claimant write attempts denied'),
    ('Art 32 Encryption', 'Field-level Fernet on NI number, sort code, account',
     None),
    ('Art 32 Key management', 'Envelope encryption, DEK wrapped by KMS-held KEK',
     'claims.dek holds only the wrapped key'),
    ('Art 30 Records of processing', 'Append-only audit log, nine mandatory fields',
     None),
    ('Art 33/34 Breach notification', 'Tamper-evident chain enables 72h scoping',
     None),
    ('EU AI Act Art 12 Record-keeping', 'Automatic event logging over system lifetime',
     'applies only if high-risk under Annex III'),
]

denied = audit_cur.execute(
    "SELECT COUNT(*) c FROM audit_log WHERE result='denied'").fetchone()['c']
total  = audit_cur.execute('SELECT COUNT(*) c FROM audit_log').fetchone()['c']
withdrawn = cur.execute(
    "SELECT COUNT(*) c FROM claims WHERE status='withdrawn'").fetchone()['c']
encrypted = cur.execute(
    'SELECT COUNT(*) c FROM claims WHERE nino_bidx IS NOT NULL').fetchone()['c']

controls[1] = (controls[1][0], controls[1][1], f'{withdrawn} claim withdrawn by the data subject')
controls[3] = (controls[3][0], controls[3][1], f'{encrypted} records, 3 fields each')
controls[5] = (controls[5][0], controls[5][1], f'{total} records, {denied} denials logged')
controls[6] = (controls[6][0], controls[6][1], 'chain verified - see Challenge')

print(f'{"Obligation":<34}{"Control":<52}evidence')
print('-' * 122)
for ob, ctrl, ev in controls:
    print(f'{ob:<34}{ctrl:<52}{ev}')

print()
print('Every row above was produced by code in this notebook. That is the')
print('difference between an architecture claim and regulatory evidence.')

Obligation                        Control                                             evidence
--------------------------------------------------------------------------------------------------------------------------
Art 5(1)(b) Purpose limitation    Role views expose only columns needed per purpose   claimant_view has no fraud_score column
Art 5(1)(e) Storage limitation    Withdrawal is a claimant-permitted transition       1 claim withdrawn by the data subject
Art 5(1)(f) + 32 Security         RLS via session views + INSTEAD OF triggers         3 of 5 claimant write attempts denied
Art 32 Encryption                 Field-level Fernet on NI number, sort code, account 7 records, 3 fields each
Art 32 Key management             Envelope encryption, DEK wrapped by KMS-held KEK    claims.dek holds only the wrapped key
Art 30 Records of processing      Append-only audit log, nine mandatory fields        9 records, 2 denials logged
Art 33/34 Breach notification     Tamper-evident chain enab

---
# Challenge - chain verification and tamper detection

### How the chain works

```
row_hash = SHA256( prev_hash + canonical_json(record) )
```

Each record binds the one before it. Edit any record and every hash after it
stops matching.

*This is a hash chain, and it predates blockchain by seventeen years: Haber and
Stornetta published it in 1991 for timestamping digital documents, and their
paper is cited in the Bitcoin whitepaper. Bitcoin added proof-of-work and
distributed consensus; the tamper-evidence itself is this one line of SHA-256.*

### Your task

1. Write `verify_audit_chain()` - walk the table, recompute every hash, report
   the first break and what kind it is.
2. Tamper with a record and watch it fire.
3. Work out why an **external anchor** is still necessary.

In [21]:
AUDIT_FIELDS = ['timestamp_utc', 'actor_id', 'on_behalf_of', 'action',
                'resource', 'before_hash', 'after_hash', 'source_ip',
                'user_agent', 'session_id', 'result', 'reason']

def verify_audit_chain(verbose=True):
    """Walk the audit log and prove it has not been altered.

    Detects three distinct problems:
      EDIT     - a record's contents changed (row_hash no longer matches)
      BREAK    - prev_hash does not point at the preceding row (delete/reorder)
      TRUNCATE - handled by the caller comparing against an external anchor
    """
    rows = audit_cur.execute('SELECT * FROM audit_log ORDER BY event_id').fetchall()
    expected_prev, problems = GENESIS, []

    for r in rows:
        if r['prev_hash'] != expected_prev:
            problems.append((r['event_id'], 'BREAK',
                             'prev_hash does not match the preceding row - '
                             'a record was deleted or reordered'))

        record = {f: r[f] for f in AUDIT_FIELDS}
        recomputed = chain_hash(r['prev_hash'], record)
        if recomputed != r['row_hash']:
            problems.append((r['event_id'], 'EDIT',
                             'contents changed after the record was written'))

        expected_prev = r['row_hash']

    if verbose:
        if not problems:
            print(f'CHAIN INTACT - {len(rows)} records verified')
            print(f'  tip: {expected_prev[:32]}...')
        else:
            print(f'CHAIN COMPROMISED - {len(problems)} problem(s) '
                  f'across {len(rows)} records')
            for eid, kind, msg in problems:
                print(f'  event_id {eid}: {kind}')
                print(f'    {msg}')
    return len(problems) == 0, expected_prev


ok, tip = verify_audit_chain()

CHAIN INTACT - 9 records verified
  tip: 5acd49b7ce4f2562c6f4c06afa29dc23...


In [22]:
# Capture the tip BEFORE tampering - this is the external anchor.
ANCHOR = tip
print(f'External anchor recorded: {ANCHOR[:32]}...')
print('(In production: a separate store, or a signed daily email.)')
print()

# --- Tamper 1: an attacker edits a record to hide a denied attempt ---------
print('=' * 68)
print('TAMPER 1 - rewrite event 4 from "denied" to "success"')
print('=' * 68)
audit_cur.execute(
    "UPDATE audit_log SET result='success', reason='approved by supervisor' "
    "WHERE event_id=4")
audit_con.commit()
verify_audit_chain()

External anchor recorded: 5acd49b7ce4f2562c6f4c06afa29dc23...
(In production: a separate store, or a signed daily email.)

TAMPER 1 - rewrite event 4 from "denied" to "success"
CHAIN COMPROMISED - 1 problem(s) across 9 records
  event_id 4: EDIT
    contents changed after the record was written


(False, '5acd49b7ce4f2562c6f4c06afa29dc23d962ec8a619edb185022f2ecbc4161d3')

In [23]:
# Restore, then try the subtler attack.
audit_cur.execute(
    "UPDATE audit_log SET result='denied', "
    "reason='policy: claimant may not approve' WHERE event_id=4")
audit_con.commit()
print('Record 4 restored.')
verify_audit_chain()

print()
print('=' * 68)
print('TAMPER 2 - delete the record entirely')
print('=' * 68)
deleted = audit_cur.execute('SELECT * FROM audit_log WHERE event_id=4').fetchone()
audit_cur.execute('DELETE FROM audit_log WHERE event_id=4')
audit_con.commit()
verify_audit_chain()

Record 4 restored.
CHAIN INTACT - 9 records verified
  tip: 5acd49b7ce4f2562c6f4c06afa29dc23...

TAMPER 2 - delete the record entirely
CHAIN COMPROMISED - 1 problem(s) across 8 records
  event_id 5: BREAK
    prev_hash does not match the preceding row - a record was deleted or reordered


(False, '5acd49b7ce4f2562c6f4c06afa29dc23d962ec8a619edb185022f2ecbc4161d3')

In [24]:
# Restore record 4 so the chain is whole again.
audit_cur.execute(
    'INSERT INTO audit_log (event_id, timestamp_utc, actor_id, on_behalf_of, '
    'action, resource, before_hash, after_hash, source_ip, user_agent, '
    'session_id, result, reason, prev_hash, row_hash) '
    'VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)',
    tuple(deleted[k] for k in deleted.keys()))
audit_con.commit()
ok, tip = verify_audit_chain()

print()
print('=' * 70)
print('TAMPER 3 - the attack the chain alone CANNOT catch')
print('=' * 70)
print('An attacker with full write access deletes the last two records AND')
print('rebuilds every hash from that point, so the chain stays internally')
print('consistent. Watch verify_audit_chain() be completely fooled.')
print()

# Delete the audit trail of the export and the PII detection.
audit_cur.execute(
    'DELETE FROM audit_log WHERE event_id IN '
    '(SELECT event_id FROM audit_log ORDER BY event_id DESC LIMIT 2)')
audit_con.commit()

# Now rebuild the chain so every hash lines up again.
rows = audit_cur.execute('SELECT * FROM audit_log ORDER BY event_id').fetchall()
prev = GENESIS
for r in rows:
    record = {f: r[f] for f in AUDIT_FIELDS}
    rh = chain_hash(prev, record)
    audit_cur.execute(
        'UPDATE audit_log SET prev_hash=?, row_hash=? WHERE event_id=?',
        (prev, rh, r['event_id']))
    prev = rh
audit_con.commit()

ok_after, tip_after = verify_audit_chain()
print()
print('The chain says INTACT. Two records are gone. Only the anchor knows:')
print(f'  anchor tip  : {ANCHOR[:40]}...')
print(f'  current tip : {tip_after[:40]}...')
print(f'  match       : {ANCHOR == tip_after}   <-- TRUNCATION DETECTED')
print()
print('Publish the tip OUTSIDE the database - a separate store, a signed daily')
print('email, or a transparency log. A hash chain proves internal consistency.')
print('An anchor proves the chain is the SAME chain.')

CHAIN INTACT - 9 records verified
  tip: 5acd49b7ce4f2562c6f4c06afa29dc23...

TAMPER 3 - the attack the chain alone CANNOT catch
An attacker with full write access deletes the last two records AND
rebuilds every hash from that point, so the chain stays internally
consistent. Watch verify_audit_chain() be completely fooled.

CHAIN INTACT - 7 records verified
  tip: 5d8aba115a8a1d6ab797d399caf23d2e...

The chain says INTACT. Two records are gone. Only the anchor knows:
  anchor tip  : 5acd49b7ce4f2562c6f4c06afa29dc23d962ec8a...
  current tip : 5d8aba115a8a1d6ab797d399caf23d2ed35612b0...
  match       : False   <-- TRUNCATION DETECTED

Publish the tip OUTSIDE the database - a separate store, a signed daily
email, or a transparency log. A hash chain proves internal consistency.
An anchor proves the chain is the SAME chain.


### Extensions, if you finish early

1. **Defeat the anchor.** TAMPER 3 was caught because the anchor was recorded
   externally. What if the attacker also controls where the anchor is stored?
   Work out what property a store must have for an anchor to be trustworthy -
   and why "append-only" alone is not quite enough.
2. **Add `INSTEAD OF` triggers to `adjuster_view`.** An adjuster may change
   status and notes, but never `claimant_id` or `nino`. Which of the
   three guards from Part C still applies, and which needs rewriting?
3. **Rotate the KEK.** Call `kms.generate_data_key()` again, re-wrap the
   existing DEK, and confirm no row of `claims` had to be re-encrypted. That
   asymmetry is the entire argument for envelope encryption.
4. **Break the blind index.** NI number format is predictable - two letters, six digits, one suffix letter. Roughly how many candidates would an attacker with the
   HMAC key need to try? What if they do not have the key?

---
# Summary

| Layer | Built in | Stops |
|---|---|---|
| 1. Row-Level Security | Parts B, C | The legitimate user who asks for rows that are not theirs |
| 2. Field-Level Encryption | Part D | The DBA, the stolen snapshot, the stray log line |
| 3. Key Management | Part D | The leaked config file that would otherwise be a data breach |
| 4. Tamper-Evident Audit | Part E, Challenge | The attacker who would rather you could not prove it |
| 5. Compliance Mapping | Part F | The ICO and the FCA, who ask for evidence rather than architecture |

### What this lab proved

- A filtered view scoped Rhea to 2 claims while the base table still showed 7
  - **enforcement by convention is not enforcement by the engine**
- A pooled connection that forgot to reset identity showed Rhea *Adam's*
  claims - the bug that passes every test suite
- Three of five claimant write attempts were denied by `INSTEAD OF` triggers
- The auditor sees `gAAAAABn...` where the NI number used to be, and exact-match
  lookup still works through the blind index
- Editing one audit record broke the chain at that exact row; deleting one
  broke it at the next
- Rebuilding the whole chain defeats verification - and only the **external
  anchor** catches that

### The closing thought

Security at rest is not a feature you add. It is a layer cake you assume will
be eaten from the top down - which is why no single layer is ever allowed to
be load-bearing.

> Return to Capital One: 106 million records, valid credentials, nothing
> "broken". Which of the five layers would have changed the outcome?
> **Layers 2 and 3, together.** That is the point of the whole session.

---

**Next session - M6 Day 1, 29 October: Model & Supply Chain Security.**
Sandboxing with Docker and gVisor, SBOM generation with Syft, vulnerability
scanning with Grype, and MCP server trust boundaries.

In [25]:
con.close()
audit_con.close()
print('Connections closed.')
print()
print('Artefacts produced by this lab:')
for f in ('claims.db', 'claims.dek'):
    if os.path.exists(f):
        print(f'  {f:<14} {os.path.getsize(f):>8,} bytes')

Connections closed.

Artefacts produced by this lab:
  claims.db        32,768 bytes
  claims.dek          140 bytes


In [29]:
import pandas as pd

pd.read_sql_query("SELECT * FROM claims", con)



ProgrammingError: Cannot operate on a closed database.